In [1]:
import sys
from pathlib import Path

# 1. Определяем корень проекта
# (подбираем количество .parent, чтобы попасть в max_projects)
project_root = Path.cwd().parent

# 2. Добавляем корень в пути поиска модулей
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 3. Проверяем, что путь добавлен
print(f"✅ Project root: {project_root}")
print(f"✅ Sys path: {sys.path[:3]}...")

✅ Project root: c:\Users\123\Desktop\start_vector
✅ Sys path: ['c:\\Users\\123\\Desktop\\start_vector', 'C:\\Python314\\python314.zip', 'C:\\Python314\\DLLs']...


In [2]:
import logging
import pandas as pd
import aiohttp
import asyncio
from datetime import datetime, timedelta
import requests
import json
import time

from src_oop.core.scraper import HTTPClient
from src_oop.core.utils_general import load_api_tokens, load_sima_land_tokens

In [ ]:
logger = logging.getLogger(__name__)

class Measurements:
    def __init__(self, date_from: str = None, date_to: str = None):
        self.url = "https://seller-analytics-api.wildberries.ru/api/analytics/v1/warehouse-measurements"

        # Устанавливаем даты по умолчанию, если они не переданы
        if date_from is None:
            self.date_from = (datetime.now() - timedelta(days=30)).strftime("%Y-%m-%dT%H:%M:%SZ")
            self.date_to = (datetime.now()).strftime("%Y-%m-%dT%H:%M:%SZ")
        else:
            self.date_from = date_from
            self.date_to = date_to

        self.delay = 60.1  # Лимит: 1 запрос в минуту
        self.retries = 10

    async def get_measurements(self, account: str, token: str, session: aiohttp.ClientSession):
        """Асинхронный генератор для получения данных о замерах"""
        
        # Создаем экземпляр HTTPClient с учетом задержки и количества повторов
        client = HTTPClient(
            session=session,
            api_key=token,
            timeout=30.0 
        )

        offset = 0
        limit = 1000
        # Постранично запрашиваем данные, пока не получим все
        while True:
            params = {
                "dateFrom": self.date_from,
                "dateTo": self.date_to,
                "limit": limit,
                "offset": offset
            }

            try:
                # Делаем запрос через HTTPClient
                response = await client.get(self.url, params=params, delay=self.delay, retries=self.retries)
            except Exception as e:
                logger.error(f"Ошибка запроса для {account}: {e}")
                break

            if not response or 'data' not in response:
                logger.warning(f"Пустой ответ или ошибка формата для {account}")
                break

            # Извлекаем список отчетов
            reports = response.get('data', {}).get('reports', [])
            
            if not reports:
                logger.info(f"Все данные для {account} получены. Итого: {offset}")
                break

            yield reports  # Возвращаем порцию данных (список объектов)
            
            # Увеличиваем смещение на количество реально полученных записей
            received_count = len(reports)
            offset += received_count
            
            logger.info(f"Аккаунт {account}: Получено {received_count} записей. Всего: {offset}")

            # Если получили меньше, чем просили (limit), значит данных больше нет
            if received_count < limit:
                break